# GPU FIT

Import

In [ ]:
import os
# import packages

import sys
sys.path.insert(1, os.path.abspath('..'))  # repo root
import SMLM
import importlib
importlib.reload(SMLM)
from SMLM import *
import pims
from glob import glob
biplane = False

In [ ]:
# import images

#pth="./synthetic testdata/"

#pth = r'\\ces-1.i-med.ac.at\imed\Biomedizinische_Physik\groups\SMLM\STORM stacks\Nanorulers_80nm\Stack_2023_08_23\take_1_45_220mw_25ms'
#name = 'take_1_45_220mw_25ms'
#filename = sorted(glob(f"{pth}\\take_1_45_220mw_25ms.tif"))


pth = r'\\ces-1.i-med.ac.at\imed\Biomedizinische_Physik\groups\SMLM\synthetic data Julia'
name = 'take_1_45_220mw_25ms'
filename = sorted(glob(f"{pth}\\data.tif"))

vid = [pims.open(n) for n in filename]
stack = np.array(vid[0])

if biplane: 
    filename2 = sorted(glob(pth + "spiral_8fold_dz=10nm_R=100nm_s5000_bg20.tif")) #for biplane imaging
    vid2 = [pims.open(n) for n in filename2]
    stack = np.concatenate((stack, np.array(vid2[0])), axis = 2)

#imshow(stack[0,:,:])

In [ ]:
# import aberrations

import mat73 as mat
path = r'\\ces-1.i-med.ac.at\imed\Biomedizinische_Physik\groups\SMLM\STORM stacks\Nanorulers_80nm\Stack_2023_08_23'
name = 'aberr_cyl'
file_name = '\\' + name + '.mat'
file_to_load = path + file_name
prl = mat.loadmat(file_to_load)
aberr_index = prl['aberr']['Z_modes']
aberr_value = prl['aberr']['Z_magn']
apodization = prl['aberr']['T_coefs']
del mat, file_to_load

Initialize camera and objective

In [ ]:
# cam and obj
cam = Camera(pixsize = 6500, # nm
             QE = 0.75, 
             baseline = 100, 
             gain = 0.21, 
             EM_gain = 1, 
             readnoise = 1)

obj = Objective(M = 60, 
                NA = 1.49, 
                RI = 1.52)

Preloc

In [ ]:
image = stack[100,...]
sigma = 1           # sigma...sigma for Gaussian filter to smooth the raw images
min_dist = 7        # min_dist...minimum distance between molecules in pixels
thresh = 2          # thresh...threshold for peak detection: a molecule image rejected if max(image) < thresh * min(image)
N_crop=13           # pixel size of the cropped image
preloc_setup(image, sigma=sigma, min_dist=min_dist, thresh=thresh, N_crop=N_crop)

In [ ]:
preloc_table, I_crop = preloc_and_crop(stack, sigma=sigma, min_dist=min_dist, thresh=thresh, N_crop = h.xr.shape[0]//h.os)
I_photons = prepare_images_for_fitting(I_crop, cam) #convert camera images from counts to photons

In [ ]:
show_prelocs(I_photons, n=24)

Initialize PSF

In [ ]:
index = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 22, 37]
aberr = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# psf
h = psf(z_min = 0,
        z_max = 500,
        z_defocus = 400,
        Nk = 128,
        Nx = 15, 
        ux = cam.pixsize/obj.M,
        uz = 5,
        Z_modes = index,
        Z_magn = aberr,
        os=2,
        SA_correction = False)
print('Aberrations magnitude:', round(h.aberr_magn/2/np.pi,3), 'rad')
#h.show_psf_xy()  # plot the psf xy section
#h.show_psf_z()   # plot the psf z section
#h.show_pupil()   # plot the pupil phase and apodization
#h.show_CRLB(cam)

In [ ]:
h.ux

Precise loc

In [ ]:
import importlib
import SMLM ; importlib.reload(SMLM)
from SMLM import *

In [ ]:
results, fit_results = perform_fit(I_photons, preloc_table, h, batch_size = 1000)

In [ ]:
results = add_CRLB(h, cam, results)
results

In [ ]:
#FILTERING & VISUALIZATION
pxl = np.inf
filtered_results, filtered_fit_results = filt_results(results, fit_results,
            x_min = -pxl, x_max = pxl,
            y_min = -pxl, y_max = pxl,
            z_min = -np.inf, z_max = np.inf,
            signal_min = -100, signal_max = 40000,
            LLR_max = np.inf)

show_results(results, fit_results,
             print_results = True,
             plot_3d = True,
             plot_2d = True,
             plot_1d = True, ms = 2,#0.2, 
             plot_hist = True, bins = 40)

Save Data

In [ ]:
name = 'take_1_45_220mw_25ms'
file_to_save = pth + '\\fitgpu_' + name + '.csv'

results.to_csv(file_to_save, index=False)
